In [1]:
import gym
import numpy as np
import gym_mppt
from gym import envs

import os
models_dir = "models/PPORec"
logdir = "logs"

if not os.path.exists(models_dir):
    os.makedirs(models_dir)
    
if not os.path.exists(logdir):
    os.makedirs(logdir)  

In [2]:
env = gym.make("mppt_shaded-v4")

num_states = env.observation_space.shape[0]
print("Size of State Space ->  {}".format(num_states))
print(env.observation_space)

print()

num_actions = env.action_space.shape[0]
print("Size of Action Space ->  {}".format(num_actions))
print(env.action_space)

# upper_bound = env.action_space.high[0]
# lower_bound = env.action_space.low[0]

# print("Max Value of Action ->  {}".format(upper_bound))
# print("Min Value of Action ->  {}".format(lower_bound))

Size of State Space ->  3
Box([  0.   0. -inf], [212.  inf  inf], (3,), float32)

Size of Action Space ->  1
Size of Action Space ->  Box([-10.], [10.], (1,), float32)


C:\Users\archi\anaconda3\envs\BTPtorch\lib\site-packages\gym\spaces\box.py:73: UserWarning: WARN: Box bound precision lowered by casting to float32
  logger.warn(


In [3]:
from sb3_contrib import RecurrentPPO
from stable_baselines3.common.evaluation import evaluate_policy
from stable_baselines3.common.env_util import make_vec_env

In [4]:
# Parallel environments
env = make_vec_env("mppt_shaded-v4", n_envs=32)
# TIMESTEPS = 10000000    # 4 hours X
# TIMESTEPS = 1000000   # 2.5 hours at 32 envs 
TIMESTEPS = 100000   # 15 minutes at 32 envs

tensorboard_log = os.path.join(logdir, "tensorboard")

model = RecurrentPPO("MlpLstmPolicy", env, verbose=1, learning_rate=0.001, tensorboard_log=tensorboard_log)
# model = PPO("MultiInputPolicy", env, verbose=1, learning_rate=0.001, tensorboard_log=tensorboard_log)
model.learn(total_timesteps=TIMESTEPS, progress_bar=True, reset_num_timesteps=False, tb_log_name="PPORec2")
# model.save(f"{models_dir}/{TIMESTEPS}")

C:\Users\archi\anaconda3\envs\BTPtorch\lib\site-packages\gym\spaces\box.py:73: UserWarning: WARN: Box bound precision lowered by casting to float32
  logger.warn(


Using cuda device
Logging to logs\tensorboard\PPORec2_0


Output()

-----------------------------
| time/              |      |
|    fps             | 3391 |
|    iterations      | 1    |
|    time_elapsed    | 1    |
|    total_timesteps | 4096 |
-----------------------------
-----------------------------------------
| rollout/                |             |
|    ep_len_mean          | 200         |
|    ep_rew_mean          | 3.41        |
| time/                   |             |
|    fps                  | 121         |
|    iterations           | 2           |
|    time_elapsed         | 67          |
|    total_timesteps      | 8192        |
| train/                  |             |
|    approx_kl            | 0.008864345 |
|    clip_fraction        | 0.122       |
|    clip_range           | 0.2         |
|    entropy_loss         | -1.43       |
|    explained_variance   | -0.219      |
|    learning_rate        | 0.001       |
|    loss                 | 0.0138      |
|    n_updates            | 10          |
|    policy_gradient_loss | -0.009

------------------------------------------
| rollout/                |              |
|    ep_len_mean          | 200          |
|    ep_rew_mean          | 40.1         |
| time/                   |              |
|    fps                  | 110          |
|    iterations           | 11           |
|    time_elapsed         | 407          |
|    total_timesteps      | 45056        |
| train/                  |              |
|    approx_kl            | 0.0011005295 |
|    clip_fraction        | 0.0185       |
|    clip_range           | 0.2          |
|    entropy_loss         | -1.38        |
|    explained_variance   | 0.142        |
|    learning_rate        | 0.001        |
|    loss                 | 2.87         |
|    n_updates            | 100          |
|    policy_gradient_loss | -0.000529    |
|    std                  | 0.957        |
|    value_loss           | 5.76         |
------------------------------------------
-------------------------------------------
| rollout/

------------------------------------------
| rollout/                |              |
|    ep_len_mean          | 200          |
|    ep_rew_mean          | 45.4         |
| time/                   |              |
|    fps                  | 106          |
|    iterations           | 20           |
|    time_elapsed         | 772          |
|    total_timesteps      | 81920        |
| train/                  |              |
|    approx_kl            | 0.0011873667 |
|    clip_fraction        | 0.0197       |
|    clip_range           | 0.2          |
|    entropy_loss         | -1.39        |
|    explained_variance   | 0.305        |
|    learning_rate        | 0.001        |
|    loss                 | 4.59         |
|    n_updates            | 190          |
|    policy_gradient_loss | 9.79e-05     |
|    std                  | 0.971        |
|    value_loss           | 10.4         |
------------------------------------------
------------------------------------------
| rollout/ 

In [5]:
model.save("ppo_recurrent_v1")
model = RecurrentPPO.load("ppo_recurrent_v1")

In [8]:
env = model.get_env()
mean_reward, std_reward = evaluate_policy(model, env, n_eval_episodes=20, warn=False)
print(mean_reward)

AttributeError: 'NoneType' object has no attribute 'unwrapped'

In [7]:
obs = env.reset()
# cell and hidden state of the LSTM
lstm_states = None
num_envs = 32
# Episode start signals are used to reset the lstm states
episode_starts = np.ones((num_envs,), dtype=bool)
for i in range(200):
    action, lstm_states = model.predict(obs, state=lstm_states, episode_start=episode_starts, deterministic=True)
    obs, rewards, dones, info = env.step(action)
    print(obs)
    print(rewards)
    episode_starts = dones

AttributeError: 'NoneType' object has no attribute 'reset'